<a href="https://colab.research.google.com/github/drfperez/openair/blob/main/SIVICtidying.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ──────────────────────────────────────────────────────────────
# SCRIPT COMPLET: Prepara dades SIVIC amb incidència
# ──────────────────────────────────────────────────────────────

library(tidyverse)

# 1️⃣ Carregar dades
sivic <- read_csv("sivic.csv")

# 2️⃣ Crear columna de districte i convertir dates
sivic <- sivic %>%
    mutate(
        districte = str_extract(nom_abs, "\\d{2}"),  # extreu els 2 primers números
        setmana_epidemiologica = as.numeric(setmana_epidemiologica)
    )

# 3️⃣ Ordenar dades
sivic <- sivic %>%
    arrange(setmana_epidemiologica, districte, codi_abs, diagnostic, sexe, grup_edat)

# 4️⃣ Agregació per districte i diagnòstic
resum_districte <- sivic %>%
    group_by(setmana_epidemiologica, districte, diagnostic) %>%
    summarise(
        casos_total = sum(casos, na.rm = TRUE),
        poblacio_total = sum(poblacio, na.rm = TRUE),
        .groups = "drop"
    )

# 5️⃣ Agregació total Barcelona
resum_barcelona <- sivic %>%
    group_by(setmana_epidemiologica, diagnostic) %>%
    summarise(
        casos_total = sum(casos, na.rm = TRUE),
        poblacio_total = sum(poblacio, na.rm = TRUE),
        .groups = "drop"
    ) %>%
    mutate(districte = "Barcelona")

# 6️⃣ Combinar districte + total Barcelona
resum_combined <- bind_rows(resum_districte, resum_barcelona)

# 7️⃣ Crear format wide per diagnòstics
resum_wide <- resum_combined %>%
    pivot_wider(
        names_from = diagnostic,
        values_from = casos_total,
        values_fill = 0
    ) %>%
    arrange(setmana_epidemiologica, districte)

# 8️⃣ Afegir incidència per 100.000 habitants per cada diagnòstic
# Primer, obtindre els noms dels diagnòstics
diagnostics <- setdiff(names(resum_wide), c("setmana_epidemiologica", "districte", "poblacio_total"))

# Crear columnes incidència
for (diag in diagnostics) {
    incid_col <- paste0(diag, "_incidencia")
    resum_wide[[incid_col]] <- (resum_wide[[diag]] / resum_wide$poblacio_total) * 100000
}

# 9️⃣ Exportar CSV final combinat amb incidència
write_csv(resum_wide, "sivic_resum_wide_combinat_incidencia.csv")

# ──────────────────────────────────────────────────────────────
# FI DEL SCRIPT
# ───